In [0]:
"""
AeroPulse Enterprise Lakehouse Platform.

Synthetic aircraft source generator.

This module simulates aircraft master data received
from the AeroPulse ERP source system.
"""

from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F


def generate_aircraft(
    spark: SparkSession,
    record_count: int = 100,
) -> DataFrame:
    """
    Generate synthetic aircraft master data.

    Parameters
    ----------
    spark:
        Active SparkSession.
    record_count:
        Number of aircraft records to generate.

    Returns
    -------
    DataFrame
        Synthetic aircraft source data.
    """

    df = (
        spark.range(1, record_count + 1)
        .withColumn(
            "aircraft_id",
            F.format_string("AC%08d", F.col("id"))
        )
        .withColumn(
            "aircraft_registration",
            F.concat(
                F.lit("VT-"),
                F.format_string("%05d", F.col("id"))
            )
        )
        .withColumn(
            "aircraft_model",
            F.when(
                F.col("id") % 4 == 0,
                F.lit("A350")
            )
            .when(
                F.col("id") % 4 == 1,
                F.lit("B787")
            )
            .when(
                F.col("id") % 4 == 2,
                F.lit("A320")
            )
            .otherwise(
                F.lit("B777")
            )
        )
        .withColumn(
            "manufacturer",
            F.when(
                F.col("id") % 2 == 0,
                F.lit("Airbus")
            )
            .otherwise(
                F.lit("Boeing")
            )
        )
        .withColumn(
            "manufacture_year",
            (
                F.lit(2015)
                + (F.col("id") % 10)
            ).cast("integer")
        )
        .withColumn(
            "airline_id",
            F.format_string(
                "AL%06d",
                ((F.col("id") - 1) % 100) + 1
            )
        )
        .withColumn(
            "engine_count",
            F.when(
                F.col("aircraft_model").isin(
                    "A350",
                    "B787",
                    "B777"
                ),
                F.lit(2)
            )
            .otherwise(F.lit(2))
        )
        .withColumn(
            "aircraft_status",
            F.when(
                F.col("id") % 20 == 0,
                F.lit("MAINTENANCE")
            )
            .otherwise(
                F.lit("ACTIVE")
            )
        )
        .drop("id")
    )

    return df